# MV-Swin-T: Mammogram Multi-Task Classification
### Fixed Version — 4 improvements:
1. ✅ `.cuda()` hardcoded dihapus → device-agnostic
2. ✅ Class weighting lebih balanced (sqrt-inverse + clamping)
3. ✅ Validation loop per epoch + early stopping
4. ✅ Pretrained Swin-T weights dari `timm` (ImageNet)

In [ ]:
import shutil, os

shutil.copy(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/dataset_classification_vindr.py',
    '/kaggle/working/dataset_classification_vindr.py'
)

if os.path.exists('/kaggle/working/models'):
    shutil.rmtree('/kaggle/working/models')

shutil.copytree(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/models',
    '/kaggle/working/models'
)

print("✅ File code berhasil di-copy ke /kaggle/working/")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader
import time
import numpy as np
from sklearn import metrics
from sklearn.preprocessing import label_binarize


In [ ]:
from dataset_classification_vindr import MakeDataset_VinDr_classification
from models.mvswintransformer import MVSwinTransformer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))
    print("Available GPUs:", torch.cuda.device_count())

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────
extension   = ".png"
target_size = 224
window_size = 7

batch_size    = 16
learning_rate = 1e-4
epochs        = 50
weight_decay  = 1e-3
alpha         = 1.0   # bobot loss BI-RADS
beta          = 1.0   # bobot loss Density

# Early stopping: berhenti jika val_loss tidak membaik selama N epoch
EARLY_STOP_PATIENCE = 10

In [ ]:
import os

if os.path.exists("/kaggle/input"):
    image_dir     = "/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed"
    label_dir_csv = "/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv"
else:
    image_dir     = "./dataset_preprocessed"
    label_dir_csv = "./breast-level_annotations.csv"

print("Image dir :", image_dir)
print("Label CSV :", label_dir_csv)
print("Image dir exists:", os.path.exists(image_dir))
print("CSV exists:", os.path.exists(label_dir_csv))

In [ ]:
# ── DATA LOADERS ───────────────────────────────────────
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ColorJitter(brightness=0.2),
    # Normalisasi ImageNet agar cocok dengan pretrained weights
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_val_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = MakeDataset_VinDr_classification(
    image_dir=image_dir, label_dir_csv=label_dir_csv,
    transform=transform_train, mode='train', target_size=target_size
)

# ── FIX: Pisah val dari train (80% train, 20% val dari split training) ──
from torch.utils.data import random_split
val_ratio  = 0.2
val_size   = int(len(train_dataset) * val_ratio)
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
# Override transform untuk val_subset agar tidak ada augmentasi
val_subset.dataset.transform = transform_val_test

test_dataset = MakeDataset_VinDr_classification(
    image_dir=image_dir, label_dir_csv=label_dir_csv,
    transform=transform_val_test, mode='test', target_size=target_size
)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train : {train_size} | Val: {val_size} | Test: {len(test_dataset)}")

In [ ]:
# ── MODEL ──────────────────────────────────────────────
model = MVSwinTransformer(img_size=224, window_size=7).to(device)

pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params          : {pytorch_total_params  // 10**6} M")
print(f"Trainable params      : {pytorch_total_trainable // 10**6} M")

In [ ]:
# ── FIX 4: LOAD PRETRAINED SWIN-T WEIGHTS DARI FILE LOKAL ───
# Load dari file .pth yang sudah ada di Kaggle dataset (tidak perlu internet)
import glob

# Cari file .pth di semua kemungkinan path input Kaggle
pth_candidates = glob.glob('/kaggle/input/**/swin_tiny_patch4_window7_224.pth', recursive=True)
if not pth_candidates:
    pth_candidates = glob.glob('/kaggle/input/**/*.pth', recursive=True)

print("File .pth ditemukan:", pth_candidates)
assert pth_candidates, "❌ File .pth tidak ditemukan! Pastikan sudah di-add ke dataset."
pretrained_pth = pth_candidates[0]
print(f"Memuat pretrained Swin-T weights dari: {pretrained_pth}")

pretrained_sd = torch.load(pretrained_pth, map_location="cpu")
# Beberapa checkpoint timm menyimpan weights di key "model" atau "state_dict"
if isinstance(pretrained_sd, dict) and "model" in pretrained_sd:
    pretrained_sd = pretrained_sd["model"]
elif isinstance(pretrained_sd, dict) and "state_dict" in pretrained_sd:
    pretrained_sd = pretrained_sd["state_dict"]

model_sd = model.state_dict()

# Mapping: pretrained key → model keys yang relevan
# patch_embed_1 dan patch_embed_2 keduanya di-init dari patch_embed pretrained
loaded, skipped = 0, 0
new_sd = {}

for key, val in pretrained_sd.items():
    # patch embedding → inject ke kedua view
    if key.startswith("patch_embed."):
        suffix = key[len("patch_embed."):]
        for view in ["patch_embed_1", "patch_embed_2"]:
            target = f"{view}.{suffix}"
            if target in model_sd and model_sd[target].shape == val.shape:
                new_sd[target] = val
                loaded += 1
    # layers_fused di-init dari layers.2 dan layers.3 pretrained
    elif key.startswith("layers.2.") or key.startswith("layers.3."):
        suffix = key[len("layers."):]
        fused_idx = int(suffix.split(".")[0]) - 2
        rest = ".".join(suffix.split(".")[1:])
        target = f"layers_fused.{fused_idx}.{rest}"
        if target in model_sd and model_sd[target].shape == val.shape:
            new_sd[target] = val
            loaded += 1
        else:
            skipped += 1
    else:
        skipped += 1

model_sd.update(new_sd)
model.load_state_dict(model_sd)
print(f"✅ Pretrained weights injected: {loaded} keys loaded, {skipped} skipped")
del pretrained_sd


In [ ]:
# ── FIX 2: CLASS WEIGHTS YANG LEBIH BALANCED ──────────
# Gunakan sqrt-inverse weighting (lebih smooth dari 1/count)
# dan clamp agar tidak ada weight yang terlalu ekstrem
def compute_class_weights_balanced(dataset, num_classes, label_key, max_weight=10.0):
    """
    Sqrt-inverse class weights dengan clamping.
    Lebih smooth dari 1/count, mencegah weight ekstrem pada kelas sangat minoritas.
    """
    counts = torch.zeros(num_classes)
    # Akses pairs dari dataset asli (bukan subset)
    pairs = dataset.dataset.pairs if hasattr(dataset, 'dataset') else dataset.pairs
    indices = dataset.indices if hasattr(dataset, 'indices') else range(len(pairs))
    for i in indices:
        counts[pairs[i][label_key]] += 1
    
    # Sqrt-inverse: 1 / sqrt(count)
    weights = 1.0 / (counts.sqrt() + 1e-6)
    # Normalize agar mean = 1
    weights = weights / weights.mean()
    # Clamp: jangan biarkan weight terlalu besar
    weights = weights.clamp(max=max_weight)
    return weights

weights_birads  = compute_class_weights_balanced(train_subset, 5, 'label_birads').to(device)
weights_density = compute_class_weights_balanced(train_subset, 4, 'label_density').to(device)

print("Class weights BI-RADS :", weights_birads.cpu().numpy().round(3))
print("Class weights Density :", weights_density.cpu().numpy().round(3))

In [ ]:
# ── OPTIMIZER, SCHEDULER, LOSS ─────────────────────────
optimizer         = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler         = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-6
)
criterion_birads  = nn.CrossEntropyLoss(weight=weights_birads)
criterion_density = nn.CrossEntropyLoss(weight=weights_density)

print("Optimizer : AdamW")
print("Scheduler : CosineAnnealingLR")

In [ ]:
# ── FIX 3: TRAINING + VALIDATION LOOP ─────────────────
model_save_path = "/kaggle/working/best_model.pth" if os.path.exists("/kaggle/input") else "./best_model.pth"
print("Model akan disimpan di:", model_save_path)

best_val_loss      = float('inf')
early_stop_counter = 0
history            = []

for epoch in range(1, epochs + 1):
    since = time.time()
    print('-' * 60)
    print(f"Epoch [{epoch}/{epochs}]")

    # ── TRAIN ──────────────────────────────────────────
    model.train()
    train_loss_sum  = 0.0
    correct_birads  = 0
    correct_density = 0
    total           = 0

    for inputs_cc, inputs_mlo, labels_birads, labels_density in train_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        optimizer.zero_grad()
        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        loss_birads  = criterion_birads(pred_birads,   labels_birads)
        loss_density = criterion_density(pred_density, labels_density)
        total_loss   = alpha * loss_birads + beta * loss_density

        total_loss.backward()
        # Gradient clipping untuk stabilitas training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum  += total_loss.item()
        total           += labels_birads.size(0)
        correct_birads  += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density += (pred_density.argmax(dim=1) == labels_density).sum().item()

    train_loss        = train_loss_sum / len(train_loader)
    train_acc_birads  = 100 * correct_birads  / total
    train_acc_density = 100 * correct_density / total

    print(f"  [TRAIN] Loss: {train_loss:.4f} | "
          f"Acc BI-RADS: {train_acc_birads:.2f}% | "
          f"Acc Density: {train_acc_density:.2f}%")

    # ── VALIDATION ─────────────────────────────────────
    model.eval()
    val_loss_sum    = 0.0
    correct_birads  = 0
    correct_density = 0
    val_total       = 0

    with torch.no_grad():
        for inputs_cc, inputs_mlo, labels_birads, labels_density in val_loader:
            inputs_cc      = inputs_cc.float().to(device)
            inputs_mlo     = inputs_mlo.float().to(device)
            labels_birads  = labels_birads.long().to(device)
            labels_density = labels_density.long().to(device)

            pred_birads, pred_density = model(inputs_cc, inputs_mlo)

            loss_birads  = criterion_birads(pred_birads,   labels_birads)
            loss_density = criterion_density(pred_density, labels_density)
            val_loss_sum += (alpha * loss_birads + beta * loss_density).item()

            val_total       += labels_birads.size(0)
            correct_birads  += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
            correct_density += (pred_density.argmax(dim=1) == labels_density).sum().item()

    val_loss        = val_loss_sum / len(val_loader)
    val_acc_birads  = 100 * correct_birads  / val_total
    val_acc_density = 100 * correct_density / val_total
    time_elapsed    = time.time() - since
    curr_lr         = optimizer.param_groups[0]["lr"]

    print(f"  [VAL]   Loss: {val_loss:.4f} | "
          f"Acc BI-RADS: {val_acc_birads:.2f}% | "
          f"Acc Density: {val_acc_density:.2f}% | "
          f"LR: {curr_lr:.6f} | "
          f"Time: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")

    history.append({
        'epoch': epoch,
        'train_loss': train_loss, 'val_loss': val_loss,
        'train_acc_birads': train_acc_birads, 'val_acc_birads': val_acc_birads,
        'train_acc_density': train_acc_density, 'val_acc_density': val_acc_density,
    })

    scheduler.step()

    # Simpan model terbaik berdasarkan VAL LOSS (bukan train loss)
    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), model_save_path)
        print(f"  ✅ Model terbaik disimpan (val_loss: {val_loss:.4f})")
    else:
        early_stop_counter += 1
        print(f"  ⏳ Early stop counter: {early_stop_counter}/{EARLY_STOP_PATIENCE}")
        if early_stop_counter >= EARLY_STOP_PATIENCE:
            print(f"  🛑 Early stopping triggered at epoch {epoch}")
            break

print("\n✅ Training selesai!")

In [ ]:
# ── PLOT TRAINING HISTORY ──────────────────────────────
import matplotlib.pyplot as plt

epochs_list      = [h['epoch'] for h in history]
train_losses     = [h['train_loss'] for h in history]
val_losses       = [h['val_loss'] for h in history]
train_acc_b      = [h['train_acc_birads'] for h in history]
val_acc_b        = [h['val_acc_birads'] for h in history]
train_acc_d      = [h['train_acc_density'] for h in history]
val_acc_d        = [h['val_acc_density'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_list, train_losses, label='Train Loss')
axes[0].plot(epochs_list, val_losses,   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_list, train_acc_b, label='Train')
axes[1].plot(epochs_list, val_acc_b,   label='Val')
axes[1].set_title('Accuracy BI-RADS'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_list, train_acc_d, label='Train')
axes[2].plot(epochs_list, val_acc_d,   label='Val')
axes[2].set_title('Accuracy Density'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print("Plot disimpan ke training_history.png")

In [ ]:
# ── EVALUASI TEST SET ──────────────────────────────────
print("=" * 60)
print("Evaluasi Test Set — Best Model")
print("=" * 60)

model.load_state_dict(torch.load(model_save_path, map_location=device))
model.eval()

correct_birads   = 0
correct_density  = 0
total            = 0
all_prob_birads  = []
all_prob_density = []
all_true_birads  = []
all_true_density = []
all_pred_birads  = []
all_pred_density = []

with torch.no_grad():
    for inputs_cc, inputs_mlo, labels_birads, labels_density in test_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        total            += labels_birads.size(0)
        correct_birads   += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density  += (pred_density.argmax(dim=1) == labels_density).sum().item()

        all_prob_birads.append(F.softmax(pred_birads,   dim=1).cpu().numpy())
        all_prob_density.append(F.softmax(pred_density, dim=1).cpu().numpy())
        all_true_birads.append(labels_birads.cpu().numpy())
        all_true_density.append(labels_density.cpu().numpy())
        all_pred_birads.append(pred_birads.argmax(dim=1).cpu().numpy())
        all_pred_density.append(pred_density.argmax(dim=1).cpu().numpy())

test_acc_birads  = 100 * correct_birads  / total
test_acc_density = 100 * correct_density / total

all_prob_birads  = np.concatenate(all_prob_birads,  axis=0)
all_prob_density = np.concatenate(all_prob_density, axis=0)
all_true_birads  = np.concatenate(all_true_birads,  axis=0)
all_true_density = np.concatenate(all_true_density, axis=0)
all_pred_birads  = np.concatenate(all_pred_birads,  axis=0)
all_pred_density = np.concatenate(all_pred_density, axis=0)

auc_birads  = metrics.roc_auc_score(
    label_binarize(all_true_birads,  classes=[0,1,2,3,4]),
    all_prob_birads,  multi_class='ovr', average='macro'
)
auc_density = metrics.roc_auc_score(
    label_binarize(all_true_density, classes=[0,1,2,3]),
    all_prob_density, multi_class='ovr', average='macro'
)

print(f"\n  BI-RADS → Accuracy: {test_acc_birads:.2f}%  | AUC (macro-OvR): {auc_birads:.4f}")
print(f"  Density → Accuracy: {test_acc_density:.2f}% | AUC (macro-OvR): {auc_density:.4f}")

print("\n── Classification Report BI-RADS ──")
print(metrics.classification_report(
    all_true_birads, all_pred_birads,
    target_names=['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5']
))

print("── Classification Report Density ──")
print(metrics.classification_report(
    all_true_density, all_pred_density,
    target_names=['Density A','Density B','Density C','Density D']
))